In [4]:
"""
================================================================================
PCA降维 + 主动学习迭代演化可视化（优化版）
================================================================================
优化内容：
    ① 新增碎石图（Scree Plot）— 证明PC选择依据
    ② 新增Biplot载荷箭头 — 显示特征对PC的贡献方向
    ③ 修复排版遮挡问题 — 上下图间距加大
    ④ 所有图片数据导出Excel，工作表注明对应图片

输出图片：
    PCA_00_scree_plot.png          ← 碎石图
    PCA_01_main_scatter.png        ← 主散点图（含载荷箭头）
    PCA_02_density_evolution.png   ← 密度演化图
    PCA_03_FINAL.png               ← 合并版（论文用）

输出Excel：
    data_01_PCA_coordinates.xlsx       ← 坐标+KQ（对应图01/02/03）
    data_02_centroids.xlsx             ← 各轮重心（对应图01/03）
    data_03_KDE_grid_origin.xlsx       ← 原始KDE网格（对应图01/03背景）
    data_04_KDE_grid_round0to5.xlsx    ← 各轮KDE网格（对应图02/03下半）
    data_05_scree_plot.xlsx            ← 碎石图数据（对应图00）
    data_06_biplot_loadings.xlsx       ← 载荷箭头数据（对应图01/03的箭头）
================================================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import gaussian_kde
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['mathtext.fontset'] = 'dejavusans'

# ============================================================
# 配置参数
# ============================================================
INPUT_FILE = (
    r"D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版"
    r"\用于UMAP降维+密度演化图-4.8.xlsx"
)
OUTPUT_DIR = (
    r"D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版"
    r"\迭代可视化_PCA_优化版"
)

FEATURE_COLS = [
    'x',
    'ΔSmix',
    'Ω',
    'Ec',
    'deltaP_E13 Electron affinity',
    'deltaP_热导率W/(mk)',
]

# 特征简称（用于Biplot箭头标注，避免太长）
FEATURE_SHORT = {
    'x':                             'χ',
    'ΔSmix':                         'ΔSmix',
    'Ω':                             'Ω',
    'Ec':                            'Ec',
    'deltaP_E13 Electron affinity':  'ΔP_EA',
    'deltaP_热导率W/(mk)':            'ΔP_TC',
}

ROUND_COLORS = {
    1: '#FF6B35',
    2: '#FFD700',
    3: '#00CC44',
    4: '#00BFFF',
    5: '#FF1493',
}
ROUND_LABELS = {
    1: 'Iteration #1',
    2: 'Iteration #2',
    3: 'Iteration #3',
    4: 'Iteration #4',
    5: 'Iteration #5',
}

KQ_CMAP  = 'RdYlGn'
KDE_CMAP = 'YlOrRd'

# ============================================================
# 0. 创建输出目录
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("=" * 65)
print("PCA降维 + 主动学习迭代演化可视化（优化版）")
print("=" * 65)
print(f"\n输出目录: {OUTPUT_DIR}")

# ============================================================
# 1. 读取数据
# ============================================================
print("\n读取数据...")
df = pd.read_excel(INPUT_FILE)
print(f"   总样本数 : {len(df)}")
print(f"   列名     : {list(df.columns)}")

df_origin = df[df['Round'] == 0].copy()
df_iter   = df[df['Round'] > 0].copy()

print(f"\n   原始样本 (Round=0): {len(df_origin)} 个")
for r in range(1, 6):
    n = len(df[df['Round'] == r])
    print(f"   Iteration #{r} (Round={r}): {n} 个")
print(f"\n   KQ 范围: [{df['KQ'].min():.2f}, {df['KQ'].max():.2f}]")

# ============================================================
# 2. PCA降维（全6维 + 前2维可视化）
# ============================================================
print("\nPCA 降维...")
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS].values)

# 全6个主成分（用于碎石图）
pca_full = PCA(n_components=6, random_state=42)
pca_full.fit(X_scaled)
var_ratio_full = pca_full.explained_variance_ratio_
cumvar_full    = np.cumsum(var_ratio_full)

# 前2个主成分（用于可视化）
pca      = PCA(n_components=2, random_state=42)
embedding = pca.fit_transform(X_scaled)
var_ratio = pca.explained_variance_ratio_

print(f"\n   全部6个PC的方差解释率：")
for i, (v, cv) in enumerate(zip(var_ratio_full, cumvar_full), 1):
    print(f"   PC{i}: {v:.1%}   累计: {cv:.1%}")

print(f"\n   载荷矩阵（Feature Loadings）：")
print(f"   {'特征':<35} {'PC1':>8}  {'PC2':>8}")
print("   " + "-" * 55)
for feat, l1, l2 in zip(FEATURE_COLS,
                          pca.components_[0],
                          pca.components_[1]):
    print(f"   {feat:<35} {l1:>8.4f}  {l2:>8.4f}")

df['PC1'] = embedding[:, 0]
df['PC2'] = embedding[:, 1]
df_origin = df[df['Round'] == 0].copy()
df_iter   = df[df['Round'] > 0].copy()

print(f"\n   完成  PC1:[{embedding[:,0].min():.2f},{embedding[:,0].max():.2f}]"
      f"  PC2:[{embedding[:,1].min():.2f},{embedding[:,1].max():.2f}]")

# ============================================================
# 3. 计算各轮重心
# ============================================================
centroids = {}
for r in range(1, 6):
    df_r = df[df['Round'] == r]
    if len(df_r) > 0:
        centroids[r] = {
            'x':       df_r['PC1'].mean(),
            'y':       df_r['PC2'].mean(),
            'kq_mean': df_r['KQ'].mean()
        }
centroid_rounds = sorted(centroids.keys())

print("\n   各轮重心:")
for r, c in centroids.items():
    print(f"      Round {r}: ({c['x']:.3f}, {c['y']:.3f})  平均KQ={c['kq_mean']:.2f}")

kq_min = df['KQ'].min()
kq_max = df['KQ'].max()
norm   = Normalize(vmin=kq_min, vmax=kq_max)

x_lim = (embedding[:,0].min() - 0.8, embedding[:,0].max() + 0.8)
y_lim = (embedding[:,1].min() - 0.8, embedding[:,1].max() + 0.8)
x_grid = np.linspace(x_lim[0], x_lim[1], 120)
y_grid = np.linspace(y_lim[0], y_lim[1], 120)
xx, yy  = np.meshgrid(x_grid, y_grid)
grid_pts = np.vstack([xx.flatten(), yy.flatten()])

high_kq_mask = df_origin['KQ'] >= np.percentile(df_origin['KQ'], 75)
hkq_x = df_origin.loc[high_kq_mask, 'PC1'].mean()
hkq_y = df_origin.loc[high_kq_mask, 'PC2'].mean()

# Biplot箭头缩放因子（让箭头长度适配散点范围）
arrow_scale = min(
    abs(x_lim[1] - x_lim[0]),
    abs(y_lim[1] - y_lim[0])
) * 0.38

# ============================================================
# 4. KDE 计算
# ============================================================
print("\n计算 KDE 密度...")
try:
    kde_origin = gaussian_kde(
        np.vstack([df_origin['PC1'], df_origin['PC2']]), bw_method=0.3
    )
    zz_origin = kde_origin(grid_pts).reshape(xx.shape)
except Exception as e:
    print(f"   原始KDE失败: {e}")
    zz_origin = np.zeros_like(xx)

kde_grids = {}
for round_num in range(0, 6):
    df_cumul = df[df['Round'] <= round_num]
    try:
        kde_r = gaussian_kde(
            np.vstack([df_cumul['PC1'], df_cumul['PC2']]),
            bw_method=0.35
        )
        zz_r = kde_r(grid_pts).reshape(xx.shape)
    except Exception as e:
        print(f"   Round {round_num} KDE失败: {e}")
        zz_r = np.zeros_like(xx)
    kde_grids[round_num] = zz_r

print("   KDE 计算完成")

# ============================================================
# 5. 导出 Excel 数据
# ============================================================
print("\n导出 Excel 数据（供 Origin 绘图）...")

# ── 5-1  PCA坐标表（对应图01/02/03）──────────────────────────
excel1_path = os.path.join(OUTPUT_DIR, "data_01_PCA_coordinates.xlsx")
df_export = df[['Sample_ID', 'Round'] + FEATURE_COLS + ['KQ', 'PC1', 'PC2']].copy()
df_export.to_excel(excel1_path, index=False, sheet_name='PCA坐标_图01_02_03主散点')
print(f"   PCA坐标表: {excel1_path}")

# ── 5-2  重心坐标表（对应图01/03箭头）────────────────────────
excel2_path = os.path.join(OUTPUT_DIR, "data_02_centroids.xlsx")
centroid_rows = []
for r, c in centroids.items():
    centroid_rows.append({
        'Round':      r,
        'Label':      ROUND_LABELS[r],
        'Centroid_PC1': c['x'],
        'Centroid_PC2': c['y'],
        'Mean_KQ':    c['kq_mean'],
        'Color':      ROUND_COLORS[r]
    })
pd.DataFrame(centroid_rows).to_excel(
    excel2_path, index=False, sheet_name='重心轨迹_图01_03箭头')
print(f"   重心坐标表: {excel2_path}")

# ── 5-3  原始KDE网格（对应图01/03背景等高线）─────────────────
excel3_path = os.path.join(OUTPUT_DIR, "data_03_KDE_grid_origin.xlsx")
pd.DataFrame({
    'X_PC1':       xx.flatten(),
    'Y_PC2':       yy.flatten(),
    'KDE_density': zz_origin.flatten()
}).to_excel(excel3_path, index=False, sheet_name='原始KDE_图01_03背景等高线')
print(f"   原始KDE密度网格: {excel3_path}")

# ── 5-4  各轮累积KDE网格（对应图02/03下半密度演化）───────────
excel4_path = os.path.join(OUTPUT_DIR, "data_04_KDE_grid_round0to5.xlsx")
sheet_names = [
    '图02_03_Training_data',
    '图02_03_After_Iter1',
    '图02_03_After_Iter2',
    '图02_03_After_Iter3',
    '图02_03_After_Iter4',
    '图02_03_After_Iter5',
]
with pd.ExcelWriter(excel4_path, engine='openpyxl') as writer:
    for round_num in range(0, 6):
        df_cumul = df[df['Round'] <= round_num]
        pd.DataFrame({
            'X_PC1':       xx.flatten(),
            'Y_PC2':       yy.flatten(),
            'KDE_density': kde_grids[round_num].flatten(),
            'n_samples':   len(df_cumul)
        }).to_excel(writer, sheet_name=sheet_names[round_num], index=False)
print(f"   各轮KDE密度网格: {excel4_path}")

# ── 5-5  碎石图数据（对应图00）───────────────────────────────
excel5_path = os.path.join(OUTPUT_DIR, "data_05_scree_plot.xlsx")
pd.DataFrame({
    'PC':                    [f'PC{i+1}' for i in range(6)],
    'Explained_Variance':    var_ratio_full,
    'Explained_Variance_pct': [f'{v:.1%}' for v in var_ratio_full],
    'Cumulative_Variance':   cumvar_full,
    'Cumulative_Variance_pct': [f'{v:.1%}' for v in cumvar_full],
    'Eigenvalue':            pca_full.explained_variance_,
}).to_excel(excel5_path, index=False, sheet_name='碎石图数据_图00')
print(f"   碎石图数据: {excel5_path}")

# ── 5-6  Biplot载荷数据（对应图01/03箭头方向）────────────────
excel6_path = os.path.join(OUTPUT_DIR, "data_06_biplot_loadings.xlsx")
loading_rows = []
for i, feat in enumerate(FEATURE_COLS):
    l1 = pca.components_[0, i]
    l2 = pca.components_[1, i]
    loading_rows.append({
        'Feature':         feat,
        'Short_name':      FEATURE_SHORT[feat],
        'Loading_PC1':     l1,
        'Loading_PC2':     l2,
        'Arrow_end_X':     l1 * arrow_scale,
        'Arrow_end_Y':     l2 * arrow_scale,
        'Arrow_length':    np.sqrt(l1**2 + l2**2) * arrow_scale,
    })
pd.DataFrame(loading_rows).to_excel(
    excel6_path, index=False, sheet_name='Biplot载荷_图01_03箭头')
print(f"   Biplot载荷数据: {excel6_path}")

# ============================================================
# 6. 绘图辅助函数
# ============================================================
PC1_LABEL = f'PC1 ({var_ratio[0]:.1%} variance explained)'
PC2_LABEL = f'PC2 ({var_ratio[1]:.1%} variance explained)'

panel_titles = [
    'Training\ndata',
    'After\nIteration #1', 'After\nIteration #2',
    'After\nIteration #3', 'After\nIteration #4', 'After\nIteration #5'
]


def draw_biplot_arrows(ax):
    """在主散点图上叠加特征载荷箭头"""
    for i, feat in enumerate(FEATURE_COLS):
        l1 = pca.components_[0, i] * arrow_scale
        l2 = pca.components_[1, i] * arrow_scale
        ax.annotate(
            '', xy=(l1, l2), xytext=(0, 0),
            arrowprops=dict(
                arrowstyle='->', color='navy',
                lw=1.8, mutation_scale=14
            ), zorder=10
        )
        offset_x = l1 * 1.18
        offset_y = l2 * 1.18
        ax.text(
            offset_x, offset_y,
            FEATURE_SHORT[feat],
            fontsize=9.5, color='navy', fontweight='bold',
            ha='center', va='center', zorder=11,
            bbox=dict(boxstyle='round,pad=0.15',
                      facecolor='white', alpha=0.7, edgecolor='none')
        )


def draw_main_scatter(ax, show_title=True, show_biplot=True):
    """主散点图"""
    sc = ax.scatter(
        df_origin['PC1'], df_origin['PC2'],
        c=df_origin['KQ'], cmap=KQ_CMAP,
        vmin=kq_min, vmax=kq_max,
        s=60, alpha=0.5, edgecolors='gray', linewidths=0.3,
        zorder=2, label=f'Training data (n={len(df_origin)})'
    )
    try:
        ax.contour(xx, yy, zz_origin, levels=5,
                   colors='gray', alpha=0.22, linewidths=0.8, zorder=1)
    except Exception:
        pass

    for r in range(1, 6):
        df_r = df[df['Round'] == r]
        if len(df_r) == 0:
            continue
        ax.scatter(
            df_r['PC1'], df_r['PC2'],
            c=ROUND_COLORS[r], s=260, marker='*',
            edgecolors='black', linewidths=0.8,
            zorder=5, label=ROUND_LABELS[r]
        )
        for _, row in df_r.iterrows():
            ax.annotate(
                f'{row["KQ"]:.1f}',
                xy=(row['PC1'], row['PC2']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=7.5, color=ROUND_COLORS[r],
                fontweight='bold', zorder=6
            )

    for i in range(len(centroid_rounds) - 1):
        rc = centroid_rounds[i];  rn = centroid_rounds[i + 1]
        cc = centroids[rc];       cn = centroids[rn]
        ax.annotate(
            '', xy=(cn['x'], cn['y']), xytext=(cc['x'], cc['y']),
            arrowprops=dict(
                arrowstyle='->,head_width=0.32,head_length=0.22',
                color='black', lw=2.1,
                connectionstyle='arc3,rad=0.15'
            ), zorder=7
        )

    for r, c in centroids.items():
        ax.plot(c['x'], c['y'], marker='D', markersize=10,
                color=ROUND_COLORS[r], markeredgecolor='black',
                markeredgewidth=1.0, zorder=8)

    if high_kq_mask.sum() > 3:
        ax.annotate(
            'High $K_Q$ region',
            xy=(hkq_x, hkq_y),
            fontsize=11, color='darkgreen', fontweight='bold', ha='center',
            bbox=dict(boxstyle='round,pad=0.3',
                      facecolor='lightgreen', alpha=0.4),
            zorder=9
        )

    if show_biplot:
        draw_biplot_arrows(ax)

    ax.set_xlabel(PC1_LABEL, fontsize=12, fontweight='bold')
    ax.set_ylabel(PC2_LABEL, fontsize=12, fontweight='bold')
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)

    if show_title:
        ax.set_title(
            'Active Learning Exploration Trajectory in PCA Feature Space',
            fontsize=13, fontweight='bold', pad=10
        )
    ax.legend(loc='upper left', fontsize=9.5, framealpha=0.85, edgecolor='gray')
    ax.grid(True, alpha=0.2, linestyle='--')
    ax.tick_params(labelsize=10)
    return sc


def draw_kde_panel(ax, round_num, col_idx):
    """KDE密度子图"""
    df_cumul = df[df['Round'] <= round_num]
    zz_r = kde_grids.get(round_num, np.zeros_like(xx))

    ax.contourf(xx, yy, zz_r, levels=15, cmap=KDE_CMAP, alpha=0.85)
    ax.contour(xx, yy, zz_r,  levels=8,  colors='white', alpha=0.4, linewidths=0.5)

    ax.scatter(df_origin['PC1'], df_origin['PC2'],
               c='white', s=7, alpha=0.35, edgecolors='none', zorder=3)

    for r in range(1, round_num + 1):
        df_r = df[df['Round'] == r]
        if len(df_r) > 0:
            ax.scatter(df_r['PC1'], df_r['PC2'],
                       c=ROUND_COLORS[r], s=110, marker='*',
                       edgecolors='black', linewidths=0.5, zorder=5)

    ax.text(0.05, 0.05, f'n={len(df_cumul)}',
            transform=ax.transAxes, fontsize=8.5,
            color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))

    ax.set_xlim(x_lim);  ax.set_ylim(y_lim)
    ax.set_title(panel_titles[col_idx], fontsize=9.5, fontweight='bold', pad=5)
    ax.set_xlabel('PC1', fontsize=8.5)
    if col_idx == 0:
        ax.set_ylabel('PC2', fontsize=8.5)
    else:
        ax.set_yticklabels([])
    ax.tick_params(labelsize=7.5)


# ============================================================
# 7. 图00：碎石图（Scree Plot）
# ============================================================
print("\n生成图00 碎石图（Scree Plot）...")
fig0, ax0 = plt.subplots(figsize=(7, 5))

x_pc = np.arange(1, 7)
bars = ax0.bar(x_pc, var_ratio_full * 100,
               color='#3A5FA8', alpha=0.80,
               edgecolor='white', linewidth=0.8,
               label='Individual variance')
ax0.plot(x_pc, cumvar_full * 100,
         color='#D4522A', marker='o', linewidth=2.2,
         markersize=7, markeredgecolor='white', markeredgewidth=1.2,
         label='Cumulative variance', zorder=5)

# 标注每个柱子的数值
for bar, v in zip(bars, var_ratio_full):
    ax0.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{v:.1%}', ha='center', va='bottom',
             fontsize=9.5, fontweight='bold', color='#3A5FA8')

# 标注累计值
for xi, cv in zip(x_pc, cumvar_full):
    ax0.text(xi + 0.15, cv * 100 + 0.8,
             f'{cv:.1%}', ha='left', va='bottom',
             fontsize=8.5, color='#D4522A')

# 80%参考线
ax0.axhline(80, color='gray', linestyle='--', linewidth=1.2, alpha=0.6)
ax0.text(6.05, 80.5, '80%', fontsize=9, color='gray')

ax0.set_xlabel('Principal Component', fontsize=12, fontweight='bold')
ax0.set_ylabel('Explained Variance (%)', fontsize=12, fontweight='bold')
ax0.set_title('Scree Plot — PCA Variance Explained',
              fontsize=13, fontweight='bold', pad=10)
ax0.set_xticks(x_pc)
ax0.set_xticklabels([f'PC{i}' for i in x_pc], fontsize=10)
ax0.set_ylim(0, 105)
ax0.legend(fontsize=10, framealpha=0.85)
ax0.grid(True, axis='y', alpha=0.25, linestyle='--')
ax0.tick_params(labelsize=10)

plt.tight_layout()
out0 = os.path.join(OUTPUT_DIR, "PCA_00_scree_plot.png")
fig0.savefig(out0, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig0)
print(f"   已保存: {out0}")

# ============================================================
# 8. 图01：主散点图（含Biplot箭头）
# ============================================================
print("\n生成图01 主散点图（含载荷箭头）...")
fig1, ax1 = plt.subplots(figsize=(10, 8))
sc1 = draw_main_scatter(ax1, show_title=True, show_biplot=True)
cbar1 = plt.colorbar(sc1, ax=ax1, pad=0.02)
cbar1.set_label('$K_Q$ (MPa·m$^{1/2}$)', fontsize=12, fontweight='bold')
cbar1.ax.tick_params(labelsize=10)
plt.tight_layout()
out1 = os.path.join(OUTPUT_DIR, "PCA_01_main_scatter.png")
fig1.savefig(out1, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig1)
print(f"   已保存: {out1}")

# ============================================================
# 9. 图02：密度演化图（1×6）
# ============================================================
print("\n生成图02 密度演化图（1×6）...")
fig2, axes2 = plt.subplots(1, 6, figsize=(22, 4.2))
for col_idx, rn in enumerate(range(0, 6)):
    draw_kde_panel(axes2[col_idx], rn, col_idx)
fig2.suptitle(
    'Exploration Space Evolution during Active Learning Iterations (PCA Space)',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
out2 = os.path.join(OUTPUT_DIR, "PCA_02_density_evolution.png")
fig2.savefig(out2, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig2)
print(f"   已保存: {out2}")

# ============================================================
# 10. 图03：合并版（论文用，修复排版遮挡）
# ============================================================
print("\n生成图03 合并版最终图（论文用）...")
fig3 = plt.figure(figsize=(18, 15))

# 上半：主散点图（底部位置从0.38→0.42，避免遮挡）
ax_top = fig3.add_axes([0.05, 0.42, 0.86, 0.54])
sc_top = draw_main_scatter(ax_top, show_title=False, show_biplot=True)
ax_top.set_title(
    'Active Learning Exploration Trajectory in PCA Feature Space',
    fontsize=13, fontweight='bold', pad=10, loc='left'
)
cbar_ax = fig3.add_axes([0.928, 0.42, 0.013, 0.54])
sm = ScalarMappable(cmap=KQ_CMAP, norm=norm)
sm.set_array([])
cb = fig3.colorbar(sm, cax=cbar_ax)
cb.set_label('$K_Q$ (MPa·m$^{1/2}$)', fontsize=12, fontweight='bold')
cb.ax.tick_params(labelsize=10)

# 下半：密度演化（6个面板，顶部位置从0.32→0.34，给x轴留空间）
n_panels  = 6
panel_w   = 0.86 / n_panels
panel_gap = 0.005
for col_idx, rn in enumerate(range(0, 6)):
    left   = 0.05 + col_idx * (panel_w + panel_gap)
    ax_bot = fig3.add_axes([left, 0.03, panel_w - panel_gap, 0.34])
    draw_kde_panel(ax_bot, rn, col_idx)

out3 = os.path.join(OUTPUT_DIR, "PCA_03_FINAL.png")
fig3.savefig(out3, dpi=300, bbox_inches='tight',
             facecolor='white', edgecolor='none')
plt.close(fig3)
print(f"   已保存: {out3}")

# ============================================================
# 11. 汇总报告
# ============================================================
print("\n" + "=" * 65)
print("全部完成！")
print("=" * 65)

print(f"\n图片文件（保存至 迭代可视化_PCA_优化版）:")
print(f"   {os.path.basename(out0)}  <- 碎石图（论文方法部分）")
print(f"   {os.path.basename(out1)}  <- 主散点图+载荷箭头")
print(f"   {os.path.basename(out2)}  <- 密度演化图")
print(f"   {os.path.basename(out3)}  <- 合并版（论文直接使用）")

print(f"\nExcel数据文件（供 Origin 重新绘图）:")
print(f"   data_01_PCA_coordinates.xlsx    — 对应图01/02/03 主散点数据")
print(f"   data_02_centroids.xlsx          — 对应图01/03 重心轨迹箭头")
print(f"   data_03_KDE_grid_origin.xlsx    — 对应图01/03 背景等高线")
print(f"   data_04_KDE_grid_round0to5.xlsx — 对应图02/03 密度演化6个面板")
print(f"   data_05_scree_plot.xlsx         — 对应图00 碎石图数据")
print(f"   data_06_biplot_loadings.xlsx    — 对应图01/03 载荷箭头坐标")

print(f"\nPCA 方差解释率（论文写作用）:")
for i, (v, cv) in enumerate(zip(var_ratio_full, cumvar_full), 1):
    print(f"   PC{i} = {v:.1%}   累计 = {cv:.1%}")

print(f"\n数据统计:")
print(f"   总样本数  : {len(df)}")
print(f"   原始样本  : {len(df_origin)}")
print(f"   迭代新增  : {len(df_iter)}")
print(f"   原始最高KQ: {df_origin['KQ'].max():.2f}")
print(f"   迭代最高KQ: {df_iter['KQ'].max():.2f}")
print(f"   KQ 提升   : +{df_iter['KQ'].max() - df_origin['KQ'].max():.2f}")
print("=" * 65)

PCA降维 + 主动学习迭代演化可视化（优化版）

输出目录: D:\博一下\第一章主动学习\主动学习结果_v7.6_方案D_LogEI_SHAP全程引导版\迭代可视化_PCA_优化版

读取数据...
   总样本数 : 217
   列名     : ['Sample_ID', 'Round', 'Nb', 'Si', 'Ti', 'Al', 'Cr', 'Hf', 'Zr', 'Mo', 'V', 'W', 'Ta', 'Sn', 'x', 'ΔSmix', 'Ω', 'Ec', 'deltaP_E13 Electron affinity', 'deltaP_热导率W/(mk)', 'KQ']

   原始样本 (Round=0): 202 个
   Iteration #1 (Round=1): 3 个
   Iteration #2 (Round=2): 3 个
   Iteration #3 (Round=3): 3 个
   Iteration #4 (Round=4): 3 个
   Iteration #5 (Round=5): 3 个

   KQ 范围: [3.40, 21.09]

PCA 降维...

   全部6个PC的方差解释率：
   PC1: 73.7%   累计: 73.7%
   PC2: 17.3%   累计: 91.0%
   PC3: 4.1%   累计: 95.1%
   PC4: 2.4%   累计: 97.5%
   PC5: 1.8%   累计: 99.2%
   PC6: 0.8%   累计: 100.0%

   载荷矩阵（Feature Loadings）：
   特征                                       PC1       PC2
   -------------------------------------------------------
   x                                    -0.1428    0.9300
   ΔSmix                                 0.4498    0.1887
   Ω                                     0.4415